# Real Data: Exact-Location Hybrid Lean vs Grid Reverse-L Column

This notebook compares the two coordinate choices directly on the same real GEMS day/month ordering.

- `Hybrid_Lean_L08F04_C4F03_Op0p063`: `keep_ori=True`, so covariance uses `Source_Latitude` / `Source_Longitude`.
- `ColumnReverseL_A4_R80_Lag2`: `keep_ori=False`, so covariance and template geometry use regular `Latitude` / `Longitude` grid coordinates.

The two models are loaded and fitted one at a time to reduce notebook memory pressure.

In [ ]:
import sys, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_vecchia_hybrid import HybridVecchiaFit
from GEMS_TCO.kernels_vecchia_grid_col_template_reuse import ReverseLColumnVecchiaFit

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64

print('SRC:', SRC)
print('torch:', torch.__version__)
print('device:', DEVICE)

In [ ]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [2]  # 0-based: [2] = July 3. Use list(range(5)) for July 1-5.
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

MM_COND_NUMBER = 100
FIT_SMOOTHS = [0.5]

# Hybrid Lean: original/exact source coordinates for covariance
HYBRID_SPEC = {
    'model': 'Hybrid_Lean_L08F04_C4F03_Op0p063_exactloc',
    'nheads': 0,
    'limit_A': 20,
    'lag1_local_count': 8,
    'lag1_fresh_count': 4,
    'lag2_local_count': 4,
    'lag2_fresh_count': 3,
    'daily_stride': 2,
    'lag1_lon_offset': 0.063,
    # None => HybridVecchiaFit builds shift lookup from the exact t0 coords in keep_ori=True data.
    'spatial_coords_for_shift': None,
}

# Reverse-L Column: regular grid coordinates only
COLUMN_SPEC = {
    'model': 'ColumnReverseL_A4_R80_Lag2_gridloc',
    'head_right_cols': 3,
    'above_count': 4,
    'right_col_count': 3,
    'right_neighbor_count': 80,
    'lag_count': 2,
    'include_lag_self': False,
    'target_chunk_size': 1024 if DEVICE.type == 'cpu' else 4096,
}
COLUMN_TEMPLATE_WARN_GT = 500

LBFGS_LR = 1.0
LBFGS_STEPS = 5
LBFGS_EVAL = 15
LBFGS_HIST = 10

INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'real_exactloc_hybrid_vs_grid_column_050526_results.csv'

print('days:', DAY_IDX_LIST)
print('output:', OUT_CSV)

In [ ]:
def phys_to_log(d):
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
            d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    p = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq':    np.exp(p[0]) / phi2,
        'range_lat':  rlon / phi3 ** 0.5,
        'range_lon':  rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat':  p[4],
        'advec_lon':  p[5],
        'nugget':     np.exp(p[6]),
    }


def make_params():
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(INIT_DICT)]


def count_valid(day_map):
    return sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in day_map.values())


def empty_device_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def hybrid_tail_count(model):
    total = 0
    for x in (getattr(model, 'X_A', None), getattr(model, 'X_AB', None), getattr(model, 'X_ABC', None)):
        if x is not None:
            total += int(x.shape[0])
    return total


def result_row_common(date_str, day_idx, smooth, model_name, kernel, keep_ori, loss, fit_iter, pre_s, fit_s, n_valid, est):
    row = {
        'date_str': date_str,
        'day_idx': day_idx,
        'smooth': smooth,
        'model': model_name,
        'kernel': kernel,
        'keep_ori': keep_ori,
        'loss': float(loss),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter) + 1,
        'precompute_s': round(pre_s, 3),
        'fit_s': round(fit_s, 3),
        'total_s': round(pre_s + fit_s, 3),
        'n_valid': int(n_valid),
    }
    row.update({f'est_{k}': est[k] for k in P_LABELS})
    return row


def column_template_diagnostics(model):
    groups = getattr(model, 'Grouped_Batches', [])
    if not groups:
        return {
            'n_templates': 0,
            'largest_template_n': 0,
            'median_template_n': 0.0,
            'mean_template_n': 0.0,
            'mean_m_by_template': 0.0,
            'median_m_by_template': 0.0,
            'max_m_by_template': 0,
        }
    group_sizes = np.asarray([int(g['target_idx'].shape[0]) for g in groups], dtype=np.int64)
    m_sizes = np.asarray([int(g['offsets'].shape[0]) for g in groups], dtype=np.int64)
    return {
        'n_templates': int(len(groups)),
        'largest_template_n': int(group_sizes.max()),
        'median_template_n': float(np.median(group_sizes)),
        'mean_template_n': float(group_sizes.mean()),
        'mean_m_by_template': float(m_sizes.mean()),
        'median_m_by_template': float(np.median(m_sizes)),
        'max_m_by_template': int(m_sizes.max()),
    }

print('Initial log params:', [round(x, 4) for x in phys_to_log(INIT_DICT)])

In [ ]:
# Load full month once.
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

sorted_month_keys = sorted(df_map.keys())
day_key_map = {
    day_idx: sorted_month_keys[day_idx * 8:(day_idx + 1) * 8]
    for day_idx in DAY_IDX_LIST
}
selected_keys = [k for day_idx in DAY_IDX_LIST for k in day_key_map[day_idx]]
df_map_selected = {k: df_map[k] for k in selected_keys}

# The grid is fixed within the month, so one selected time slot is enough for coordinates.
first_key = selected_keys[0]
first_df = df_map_selected[first_key].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)
source_coords_t0_ordered = first_df[['Source_Latitude', 'Source_Longitude']].values.astype(np.float64)

# Drop full-month dataframe map after extracting the selected day(s), reducing notebook memory pressure.
del df_map
gc.collect()

print(f'Monthly mean TCO: {monthly_mean:.4f}')
print(f'Month time slots loaded then trimmed: {len(sorted_month_keys)} -> {len(selected_keys)}')
print(f'Grid cells: {len(grid_coords_ordered):,}')
print('grid lat:', grid_coords_ordered[:, 0].min(), grid_coords_ordered[:, 0].max())
print('grid lon:', grid_coords_ordered[:, 1].min(), grid_coords_ordered[:, 1].max())
print('source selected t0 lat:', np.nanmin(source_coords_t0_ordered[:, 0]), np.nanmax(source_coords_t0_ordered[:, 0]))
print('source selected t0 lon:', np.nanmin(source_coords_t0_ordered[:, 1]), np.nanmax(source_coords_t0_ordered[:, 1]))

In [ ]:
def fit_hybrid_exact_day(day_idx, smooth):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    day_keys = day_key_map[day_idx]
    day_df_map = {k: df_map_selected[k] for k in day_keys}
    hour_indices = [0, len(day_keys)]

    # keep_ori=True: covariance coordinates are Source_Latitude/Source_Longitude.
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=True,
    )
    day_map = {k: v.to(DEVICE) for k, v in day_map.items()}
    n_valid = count_valid(day_map)

    print('\n' + '=' * 80)
    print(f'HYBRID EXACT | DAY {date_str} | smooth={smooth} | {day_keys[0]} ... {day_keys[-1]} | valid={n_valid:,}')

    params = make_params()
    model = HybridVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        nns_map=nns_map_full,
        mm_cond_number=MM_COND_NUMBER,
        nheads=HYBRID_SPEC['nheads'],
        limit_A=HYBRID_SPEC['limit_A'],
        limit_B_local=HYBRID_SPEC['lag1_local_count'],
        limit_C_local=HYBRID_SPEC['lag2_local_count'],
        daily_stride=HYBRID_SPEC['daily_stride'],
        spatial_coords=HYBRID_SPEC['spatial_coords_for_shift'],
        lag1_lon_offset=HYBRID_SPEC['lag1_lon_offset'],
        lag1_fresh_count=HYBRID_SPEC['lag1_fresh_count'],
        lag2_fresh_count=HYBRID_SPEC['lag2_fresh_count'],
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre

    optimizer = model.set_optimizer(
        params, lr=LBFGS_LR,
        max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL,
        history_size=LBFGS_HIST,
    )
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit

    est = backmap_params(out)
    row = result_row_common(
        date_str, day_idx, smooth, HYBRID_SPEC['model'], 'hybrid_lean_exactloc',
        True, float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est,
    )
    row.update({
        'total_conditioning_nominal': 20 + 1 + 8 + 4 + 1 + 4 + 3,
        'n_heads': int(model.Heads_data.shape[0]),
        'n_tails': hybrid_tail_count(model),
        'n_templates': np.nan,
        'coords_used_for_covariance': 'Source_Latitude/Source_Longitude',
        'coords_used_for_shift_lookup': 'exact_t0_source_coords',
    })
    print('RESULT:', row)

    del model, day_map, params, optimizer
    empty_device_cache()
    return row


def fit_column_grid_day(day_idx, smooth):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    day_keys = day_key_map[day_idx]
    day_df_map = {k: df_map_selected[k] for k in day_keys}
    hour_indices = [0, len(day_keys)]

    # keep_ori=False: covariance/template coordinates are regular Latitude/Longitude grid coordinates.
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=False,
    )
    day_map = {k: v.to(DEVICE) for k, v in day_map.items()}
    n_valid = count_valid(day_map)

    print('\n' + '=' * 80)
    print(f'COLUMN GRID | DAY {date_str} | smooth={smooth} | {day_keys[0]} ... {day_keys[-1]} | valid={n_valid:,}')

    params = make_params()
    model = ReverseLColumnVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        mm_cond_number=MM_COND_NUMBER,
        grid_coords=grid_coords_ordered,
        head_right_cols=COLUMN_SPEC['head_right_cols'],
        above_count=COLUMN_SPEC['above_count'],
        right_col_count=COLUMN_SPEC['right_col_count'],
        right_neighbor_count=COLUMN_SPEC['right_neighbor_count'],
        lag_count=COLUMN_SPEC['lag_count'],
        include_lag_self=COLUMN_SPEC['include_lag_self'],
        target_chunk_size=COLUMN_SPEC['target_chunk_size'],
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = column_template_diagnostics(model)
    if diag['n_templates'] > COLUMN_TEMPLATE_WARN_GT:
        print(f"WARNING: column templates exploded: {diag['n_templates']} templates. Missing pattern may be breaking reuse.")
    print('COLUMN DIAGNOSTICS:', diag)

    optimizer = model.set_optimizer(
        params, lr=LBFGS_LR,
        max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL,
        history_size=LBFGS_HIST,
    )
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit

    est = backmap_params(out)
    row = result_row_common(
        date_str, day_idx, smooth, COLUMN_SPEC['model'], 'column_reverse_l_gridloc',
        False, float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est,
    )
    row.update({
        'total_conditioning_nominal': (COLUMN_SPEC['above_count'] + COLUMN_SPEC['right_neighbor_count']) * (COLUMN_SPEC['lag_count'] + 1),
        'n_heads': int(model.Heads_data.shape[0]),
        'n_tails': int(model.n_tails),
        **diag,
        'coords_used_for_covariance': 'Latitude/Longitude grid',
        'coords_used_for_shift_lookup': 'regular_grid_template',
        'head_right_cols': COLUMN_SPEC['head_right_cols'],
        'above_count': COLUMN_SPEC['above_count'],
        'right_neighbor_count': COLUMN_SPEC['right_neighbor_count'],
        'lag_count': COLUMN_SPEC['lag_count'],
    })
    print('RESULT:', row)

    del model, day_map, params, optimizer
    empty_device_cache()
    return row

In [ ]:
# Run comparison.
# Hybrid is run first and then freed; column is loaded separately with keep_ori=False.
rows = []

for day_idx in DAY_IDX_LIST:
    for smooth in FIT_SMOOTHS:
        rows.append(fit_hybrid_exact_day(day_idx, smooth))
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
        rows.append(fit_column_grid_day(day_idx, smooth))
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

results = pd.DataFrame(rows)
print('Saved:', OUT_CSV)
display(results)

In [ ]:
summary_cols = [
    'date_str', 'smooth', 'model', 'kernel', 'keep_ori',
    'total_conditioning_nominal', 'n_heads', 'n_tails', 'n_templates',
    'largest_template_n', 'median_template_n', 'mean_template_n', 'mean_m_by_template', 'max_m_by_template',
    'loss', 'precompute_s', 'fit_s', 'total_s', 'n_valid',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'est_nugget',
    'coords_used_for_covariance', 'coords_used_for_shift_lookup',
]
existing = [c for c in summary_cols if c in results.columns]
display(results[existing].sort_values(['date_str', 'smooth', 'kernel']))

param_rows = []
for _, row in results.iterrows():
    for p in P_LABELS:
        param_rows.append({
            'date_str': row['date_str'],
            'model': row['model'],
            'parameter': p,
            'estimate': row[f'est_{p}'],
        })
param_est = pd.DataFrame(param_rows)
display(param_est)

## Why the older real column notebook may have crashed

If this notebook runs while the older one crashed, the likely cause was not simply row count. Plausible differences:

- the older notebook kept more real-data state alive while fitting;
- `df_map`, exact/grid maps, optimizer graph, and column grouped tensors may have overlapped in memory;
- a Jupyter kernel that already ran several heavy fits can have allocator fragmentation;
- the dense exact head block is still a risk factor, but high-resolution simulation showed `heads=2736` can be feasible when the state is clean.

This notebook loads only one coordinate version at a time for the fitting stage.